In [ ]:
from gitsource import GithubRepositoryDataReader
from gitsource import chunk_documents


reader = GithubRepositoryDataReader(
    repo_owner="spring-projects",
    repo_name="spring-ai",
    commit_id="main",             # or a specific commit SHA / tag like your example
    allowed_extensions={"adoc"},   # Spring AI docs use .adoc, not .md
    filename_filter=lambda
    path: "/pages/" in path,
)
files = reader.read()

documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

chunks = chunk_documents(documents, size=2000, step=1000)

print(chunks[2])

print(f"Content: {chunks[2]['content']}")
print(f"Filename: {chunks[2]['filename']}")
print(f"Start: {chunks[2]['start']}")



In [ ]:
from elasticsearch import Elasticsearch
    
es = Elasticsearch("http://localhost:9200")

response = es.search(
    index="spring-ai-docs",
    query={"match_all": {}},
    size=5,
    source_excludes=["embedding"],
)

for hit in response["hits"]["hits"]:
    print(hit["_source"])

In [7]:
from elasticsearch import Elasticsearch
from sentence_transformers import SentenceTransformer



query = "What annotation can use to definea MCP server?"

es = Elasticsearch("http://localhost:9200")

model = SentenceTransformer("BAAI/bge-small-en-v1.5")
query_embedding = model.encode(
        query,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).tolist()


# BM25 search
bm25 = es.search(
        index="spring-ai-docs",
        query={
            "match": {
                "content": query
            }
        },
        size=20,
        source_excludes=["embedding"],
    )

# Vector search
knn = es.search(
        index="spring-ai-docs",
        knn={
            "field": "embedding",
            "query_vector": query_embedding,
            "k": 20,
            "num_candidates": 100,
        },
        source_excludes=["embedding"],
    )

scores = {}
docs = {}
for ranking in [bm25["hits"]["hits"], knn["hits"]["hits"]]:
    for rank, hit in enumerate(ranking):
        doc_id = hit["_id"]
        docs[doc_id] = hit
        scores.setdefault(doc_id, 0.0)
        scores[doc_id] += 1.0 / (60 + rank + 1)
ranked = sorted(
    scores.items(),
    key=lambda x: x[1],
    reverse=True,
)

print("Ranked results:")
print([
        docs[doc_id]["_source"]
        for doc_id, _ in ranked[:5]
    ])


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Ranked results:
[{'content': '    <artifactId>spring-ai-mcp-annotations</artifactId>\n</dependency>\n----\n\nThe MCP annotations are automatically included when you use any of the MCP Boot Starters:\n\n* `spring-ai-starter-mcp-client`\n* `spring-ai-starter-mcp-client-webflux`\n* `spring-ai-starter-mcp-server`\n* `spring-ai-starter-mcp-server-webflux`\n* `spring-ai-starter-mcp-server-webmvc`\n\n=== Configuration\n\nThe annotation scanning is enabled by default when using the MCP Boot Starters. You can configure the scanning behavior using the following properties:\n\n==== Client Annotation Scanner\n\n[source,yaml]\n----\nspring:\n  ai:\n    mcp:\n      client:\n        annotation-scanner:\n          enabled: true  # Enable/disable annotation scanning\n----\n\n==== Server Annotation Scanner\n\n[source,yaml]\n----\nspring:\n  ai:\n    mcp:\n      server:\n        annotation-scanner:\n          enabled: true  # Enable/disable annotation scanning\n----\n\n== Quick Example\n\nHere\'s a simpl